In [1]:
import os
print(os.getcwd())

C:\Users\Dhanshri\Desktop\EIA ERCOT PROJECT


In [2]:
import sqlite3
conn = sqlite3.connect("data/grid_stress.db")
count = conn.execute("SELECT COUNT(*) FROM fact_grid_hourly").fetchone()
span = conn.execute("SELECT MIN(hour_utc), MAX(hour_utc) FROM fact_grid_hourly").fetchone()
conn.close()
print("Total rows:", count)
print("Date range:", span)

Total rows: (92,)
Date range: ('2026-08-20T14:00:00Z', '2026-08-24T09:00:00Z')


In [3]:
import os
print(os.listdir("ingest"))

['.ipynb_checkpoints', 'db.py', 'eia_fetch.py', 'ercot_test.ipynb', 'features.py', 'gridstatus_fetch.py', 'load_to_db.py', 'pipeline.py', 'save_raw.py', 'setup.ipynb', 'test_pipeline.ipynb', 'train_model.py', 'transform.py', 'validate.py', '__init__.py', '__pycache__']


In [4]:
%%writefile ingest/features.py
import pandas as pd
import numpy as np


def load_fact_table():
    import sqlite3
    conn = sqlite3.connect("data/grid_stress.db")
    df = pd.read_sql("SELECT * FROM fact_grid_hourly ORDER BY hour_utc", conn)
    conn.close()
    df["hour_utc"] = pd.to_datetime(df["hour_utc"])
    return df


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values("hour_utc").reset_index(drop=True)

    df["hour_of_day"] = df["hour_utc"].dt.hour
    df["day_of_week"] = df["hour_utc"].dt.dayofweek
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

    df["forecast_error_mwh"] = df["demand_mwh"] - df["day_ahead_forecast_mwh"]
    df["interchange_ratio"] = df["total_interchange_mwh"] / df["demand_mwh"]

    for lag in [1, 3, 6, 24]:
        df[f"avg_lmp_lag{lag}"] = df["avg_lmp"].shift(lag)
        df[f"demand_mwh_lag{lag}"] = df["demand_mwh"].shift(lag)

    for window in [3, 24]:
        df[f"avg_lmp_roll_mean{window}"] = df["avg_lmp"].shift(1).rolling(window).mean()
        df[f"avg_lmp_roll_std{window}"] = df["avg_lmp"].shift(1).rolling(window).std()
        df[f"demand_mwh_roll_mean{window}"] = df["demand_mwh"].shift(1).rolling(window).mean()

    df["target_price_next_hour"] = df["avg_lmp"].shift(-1)

    price_stress = (df["avg_lmp"] - df["avg_lmp_roll_mean24"]) / df["avg_lmp_roll_mean24"].abs().replace(0, np.nan)
    demand_stress = (df["demand_mwh"] - df["net_generation_mwh"]) / df["demand_mwh"]
    volatility_stress = df["avg_congestion"].abs() / df["avg_congestion"].abs().rolling(24).mean().replace(0, np.nan)

    df["stress_score"] = (
        price_stress.rank(pct=True) * 100 * 0.4
        + demand_stress.rank(pct=True) * 100 * 0.35
        + volatility_stress.rank(pct=True) * 100 * 0.25
    )

    df["target_stress_next_hour"] = df["stress_score"].shift(-1)

    return df


if __name__ == "__main__":
    df = load_fact_table()
    print("Raw fact table shape:", df.shape)
    featured = engineer_features(df)
    print("Featured shape:", featured.shape)
    print(featured.tail(10))

Overwriting ingest/features.py


In [5]:
from ingest.features import load_fact_table, engineer_features

df = load_fact_table()
print("Raw shape:", df.shape)

featured = engineer_features(df)
print("Featured shape:", featured.shape)
featured.tail(10)

C:\Users\Dhanshri\Desktop\Downloads\Anaconda\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Raw shape: (92, 10)
Featured shape: (92, 32)


,id,hour_utc,demand_mwh,day_ahead_forecast_mwh,net_generation_mwh,total_interchange_mwh,avg_lmp,max_lmp,avg_congestion,created_at,...,demand_mwh_lag24,avg_lmp_roll_mean3,avg_lmp_roll_std3,demand_mwh_roll_mean3,avg_lmp_roll_mean24,avg_lmp_roll_std24,demand_mwh_roll_mean24,target_price_next_hour,stress_score,target_stress_next_hour
82,83,2026-08-24 00:00:00+00:00,40318.0,36591.0,NaN,NaN,NaN,NaN,NaN,2026-08-24T10:32:44.938200+00:00,...,38258.0,44.984591,7.544711,36920.333333,NaN,NaN,NaN,NaN,NaN,NaN
83,84,2026-08-24 01:00:00+00:00,41434.0,38846.0,NaN,NaN,NaN,NaN,NaN,2026-08-24T10:32:44.938200+00:00,...,39241.0,NaN,NaN,38736.000000,NaN,NaN,NaN,NaN,NaN,NaN
84,85,2026-08-24 02:00:00+00:00,41881.0,39801.0,NaN,NaN,NaN,NaN,NaN,2026-08-24T10:32:44.938200+00:00,...,39418.0,NaN,NaN,40250.666667,NaN,NaN,NaN,NaN,NaN,NaN
85,86,2026-08-24 03:00:00+00:00,39001.0,39001.0,NaN,NaN,NaN,NaN,NaN,2026-08-24T10:32:44.938200+00:00,...,37826.0,NaN,NaN,41211.000000,NaN,NaN,NaN,NaN,NaN,NaN
86,87,2026-08-24 04:00:00+00:00,37825.0,37825.0,NaN,NaN,NaN,NaN,NaN,2026-08-24T10:32:44.938200+00:00,...,36698.0,NaN,NaN,40772.000000,NaN,NaN,NaN,NaN,NaN,NaN
87,88,2026-08-24 05:00:00+00:00,36242.0,36242.0,NaN,NaN,NaN,NaN,NaN,2026-08-24T10:32:44.938200+00:00,...,35299.0,NaN,NaN,39569.000000,NaN,NaN,NaN,NaN,NaN,NaN
88,89,2026-08-24 06:00:00+00:00,33609.0,33609.0,NaN,NaN,NaN,NaN,NaN,2026-08-24T10:32:44.938200+00:00,...,33104.0,NaN,NaN,37689.333333,NaN,NaN,NaN,NaN,NaN,NaN
89,90,2026-08-24 07:00:00+00:00,30854.0,30854.0,NaN,NaN,NaN,NaN,NaN,2026-08-24T10:32:44.938200+00:00,...,30785.0,NaN,NaN,35892.000000,NaN,NaN,NaN,NaN,NaN,NaN
90,91,2026-08-24 08:00:00+00:00,30021.0,NaN,NaN,NaN,NaN,NaN,NaN,2026-08-24T10:32:44.938200+00:00,...,30115.0,NaN,NaN,33568.333333,NaN,NaN,NaN,NaN,NaN,NaN
91,92,2026-08-24 09:00:00+00:00,28360.0,NaN,NaN,NaN,NaN,NaN,NaN,2026-08-24T10:32:44.938200+00:00,...,28181.0,NaN,NaN,31494.666667,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
%%writefile ingest/train_model.py
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

from ingest.features import load_fact_table, engineer_features

FEATURE_COLS = [
    "hour_of_day", "day_of_week", "is_weekend",
    "forecast_error_mwh", "interchange_ratio",
    "avg_lmp_lag1", "avg_lmp_lag3", "avg_lmp_lag6", "avg_lmp_lag24",
    "demand_mwh_lag1", "demand_mwh_lag3", "demand_mwh_lag6", "demand_mwh_lag24",
    "avg_lmp_roll_mean3", "avg_lmp_roll_std3",
    "avg_lmp_roll_mean24", "avg_lmp_roll_std24",
    "demand_mwh_roll_mean3", "demand_mwh_roll_mean24",
]


def prepare_training_data(target_col: str):
    """
    Loads, engineers features, and drops rows where the target
    or any required feature is missing (due to source reporting lag).
    """
    df = load_fact_table()
    featured = engineer_features(df)

    required_cols = FEATURE_COLS + [target_col]
    clean = featured.dropna(subset=required_cols).reset_index(drop=True)

    X = clean[FEATURE_COLS]
    y = clean[target_col]
    return X, y, clean


def train_and_evaluate(target_col: str, label: str):
    X, y, clean = prepare_training_data(target_col)

    print(f"\n=== {label} ===")
    print(f"Usable rows after dropping NaN: {len(X)}")

    if len(X) < 20:
        print("WARNING: very few usable rows - metrics will be unstable. "
              "Treat this as a v1 baseline, retrain once more data accumulates.")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, shuffle=False  # time-series: no shuffling
    )

    # Baseline: naive persistence (predict = last known value = lag1)
    baseline_pred = X_test["avg_lmp_lag1"] if "avg_lmp_lag1" in X_test else X_test.iloc[:, 0]
    baseline_mae = mean_absolute_error(y_test, X_test["avg_lmp_lag1"])

    model = GradientBoostingRegressor(
        n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    model_mae = mean_absolute_error(y_test, preds)
    model_rmse = mean_squared_error(y_test, preds) ** 0.5

    print(f"Naive baseline MAE:  {baseline_mae:.2f}")
    print(f"Model MAE:           {model_mae:.2f}")
    print(f"Model RMSE:          {model_rmse:.2f}")
    print(f"Improvement over baseline: {(1 - model_mae/baseline_mae)*100:.1f}%")

    importances = pd.Series(model.feature_importances_, index=FEATURE_COLS)
    print("\nTop 5 features:")
    print(importances.sort_values(ascending=False).head(5))

    return model, {"baseline_mae": baseline_mae, "model_mae": model_mae, "model_rmse": model_rmse}


if __name__ == "__main__":
    price_model, price_metrics = train_and_evaluate("target_price_next_hour", "Price Forecast")
    joblib.dump(price_model, "data/price_model.pkl")

    stress_model, stress_metrics = train_and_evaluate("target_stress_next_hour", "Stress Score Forecast")
    joblib.dump(stress_model, "data/stress_model.pkl")

Overwriting ingest/train_model.py


In [7]:
%pip install scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [8]:
from ingest.train_model import train_and_evaluate
import joblib

price_model, price_metrics = train_and_evaluate("target_price_next_hour", "Price Forecast")
joblib.dump(price_model, "data/price_model.pkl")

stress_model, stress_metrics = train_and_evaluate("target_stress_next_hour", "Stress Score Forecast")
joblib.dump(stress_model, "data/stress_model.pkl")


=== Price Forecast ===
Usable rows after dropping NaN: 0


ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [1]:
%%writefile ingest/train_model.py
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

from ingest.features import load_fact_table, engineer_features

FEATURE_COLS = [
    "hour_of_day", "day_of_week", "is_weekend",
    "forecast_error_mwh", "interchange_ratio",
    "avg_lmp_lag1", "avg_lmp_lag3", "avg_lmp_lag6", "avg_lmp_lag24",
    "demand_mwh_lag1", "demand_mwh_lag3", "demand_mwh_lag6", "demand_mwh_lag24",
    "avg_lmp_roll_mean3", "avg_lmp_roll_std3",
    "avg_lmp_roll_mean24", "avg_lmp_roll_std24",
    "demand_mwh_roll_mean3", "demand_mwh_roll_mean24",
]


def prepare_training_data(target_col: str):
    df = load_fact_table()
    featured = engineer_features(df)
    clean = featured.dropna(subset=[target_col]).reset_index(drop=True)
    X = clean[FEATURE_COLS]
    y = clean[target_col]
    return X, y, clean


def train_and_evaluate(target_col: str, label: str):
    X, y, clean = prepare_training_data(target_col)

    print(f"\n=== {label} ===")
    print(f"Usable rows (target present): {len(X)}")
    print(f"Feature completeness:\n{X.notna().mean().round(2)}")

    if len(X) < 20:
        print("WARNING: very few usable rows - metrics will be unstable. "
              "Treat this as a v1 baseline, retrain once more data accumulates.")

    if len(X) < 5:
        print("Not enough data to train yet. Skipping.")
        return None, None

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, shuffle=False
    )

    baseline_series = X_test["avg_lmp_lag1"].ffill().bfill()
    baseline_mae = mean_absolute_error(y_test, baseline_series)

    model = HistGradientBoostingRegressor(
        max_iter=100, max_depth=3, learning_rate=0.1, random_state=42
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    model_mae = mean_absolute_error(y_test, preds)
    model_rmse = mean_squared_error(y_test, preds) ** 0.5

    print(f"Naive baseline MAE:  {baseline_mae:.2f}")
    print(f"Model MAE:           {model_mae:.2f}")
    print(f"Model RMSE:          {model_rmse:.2f}")
    print(f"Improvement over baseline: {(1 - model_mae/baseline_mae)*100:.1f}%")

    return model, {"baseline_mae": baseline_mae, "model_mae": model_mae, "model_rmse": model_rmse}


if __name__ == "__main__":
    price_model, price_metrics = train_and_evaluate("target_price_next_hour", "Price Forecast")
    if price_model:
        joblib.dump(price_model, "data/price_model.pkl")

    stress_model, stress_metrics = train_and_evaluate("target_stress_next_hour", "Stress Score Forecast")
    if stress_model:
        joblib.dump(stress_model, "data/stress_model.pkl")

Overwriting ingest/train_model.py


In [2]:
from ingest.train_model import train_and_evaluate
import joblib

price_model, price_metrics = train_and_evaluate("target_price_next_hour", "Price Forecast")
if price_model:
    joblib.dump(price_model, "data/price_model.pkl")

stress_model, stress_metrics = train_and_evaluate("target_stress_next_hour", "Stress Score Forecast")
if stress_model:
    joblib.dump(stress_model, "data/stress_model.pkl")

C:\Users\Dhanshri\Desktop\Downloads\Anaconda\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED



=== Price Forecast ===
Usable rows (target present): 31
Feature completeness:
hour_of_day               1.00
day_of_week               1.00
is_weekend                1.00
forecast_error_mwh        0.90
interchange_ratio         0.29
avg_lmp_lag1              0.77
avg_lmp_lag3              0.52
avg_lmp_lag6              0.16
avg_lmp_lag24             0.65
demand_mwh_lag1           0.87
demand_mwh_lag3           0.81
demand_mwh_lag6           0.74
demand_mwh_lag24          0.68
avg_lmp_roll_mean3        0.52
avg_lmp_roll_std3         0.52
avg_lmp_roll_mean24       0.00
avg_lmp_roll_std24        0.00
demand_mwh_roll_mean3     0.61
demand_mwh_roll_mean24    0.06
dtype: float64
Naive baseline MAE:  7.60
Model MAE:           9.34
Model RMSE:          10.44
Improvement over baseline: -22.9%

=== Stress Score Forecast ===
Usable rows (target present): 0
Feature completeness:
hour_of_day              NaN
day_of_week              NaN
is_weekend               NaN
forecast_error_mwh       NaN
int

In [1]:
%%writefile ingest/features.py
import pandas as pd
import numpy as np


def load_fact_table():
    import sqlite3
    conn = sqlite3.connect("data/grid_stress.db")
    df = pd.read_sql("SELECT * FROM fact_grid_hourly ORDER BY hour_utc", conn)
    conn.close()
    df["hour_utc"] = pd.to_datetime(df["hour_utc"])
    return df


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values("hour_utc").reset_index(drop=True)

    df["hour_of_day"] = df["hour_utc"].dt.hour
    df["day_of_week"] = df["hour_utc"].dt.dayofweek
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

    df["forecast_error_mwh"] = df["demand_mwh"] - df["day_ahead_forecast_mwh"]
    df["interchange_ratio"] = df["total_interchange_mwh"] / df["demand_mwh"]

    for lag in [1, 3, 6, 24]:
        df[f"avg_lmp_lag{lag}"] = df["avg_lmp"].shift(lag)
        df[f"demand_mwh_lag{lag}"] = df["demand_mwh"].shift(lag)

    for window in [3, 24]:
        df[f"avg_lmp_roll_mean{window}"] = df["avg_lmp"].shift(1).rolling(window).mean()
        df[f"avg_lmp_roll_std{window}"] = df["avg_lmp"].shift(1).rolling(window).std()
        df[f"demand_mwh_roll_mean{window}"] = df["demand_mwh"].shift(1).rolling(window).mean()

    df["target_price_next_hour"] = df["avg_lmp"].shift(-1)

    # --- Stress score: using SHORTER 6hr windows for now ---
    # (v1: current data history is too sparse for stable 24hr windows;
    #  revisit and widen to 24hr once pipeline has 1-2+ weeks of continuous data)
    STRESS_WINDOW = 6

    roll_mean_stress = df["avg_lmp"].shift(1).rolling(STRESS_WINDOW, min_periods=3).mean()
    price_stress = (df["avg_lmp"] - roll_mean_stress) / roll_mean_stress.abs().replace(0, np.nan)

    demand_stress = (df["demand_mwh"] - df["net_generation_mwh"]) / df["demand_mwh"]

    congestion_roll = df["avg_congestion"].abs().rolling(STRESS_WINDOW, min_periods=3).mean()
    volatility_stress = df["avg_congestion"].abs() / congestion_roll.replace(0, np.nan)

    df["stress_score"] = (
        price_stress.rank(pct=True) * 100 * 0.4
        + demand_stress.rank(pct=True) * 100 * 0.35
        + volatility_stress.rank(pct=True) * 100 * 0.25
    )

    df["target_stress_next_hour"] = df["stress_score"].shift(-1)

    return df


if __name__ == "__main__":
    df = load_fact_table()
    print("Raw fact table shape:", df.shape)
    featured = engineer_features(df)
    print("Featured shape:", featured.shape)
    print(featured.tail(10))

Overwriting ingest/features.py


In [2]:
import importlib
import ingest.features
importlib.reload(ingest.features)

import ingest.train_model
importlib.reload(ingest.train_model)
from ingest.train_model import train_and_evaluate

import joblib

price_model, price_metrics = train_and_evaluate("target_price_next_hour", "Price Forecast")
if price_model:
    joblib.dump(price_model, "data/price_model.pkl")

stress_model, stress_metrics = train_and_evaluate("target_stress_next_hour", "Stress Score Forecast")
if stress_model:
    joblib.dump(stress_model, "data/stress_model.pkl")

C:\Users\Dhanshri\Desktop\Downloads\Anaconda\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED



=== Price Forecast ===
Usable rows (target present): 31
Feature completeness:
hour_of_day               1.00
day_of_week               1.00
is_weekend                1.00
forecast_error_mwh        0.90
interchange_ratio         0.29
avg_lmp_lag1              0.77
avg_lmp_lag3              0.52
avg_lmp_lag6              0.16
avg_lmp_lag24             0.65
demand_mwh_lag1           0.87
demand_mwh_lag3           0.81
demand_mwh_lag6           0.74
demand_mwh_lag24          0.68
avg_lmp_roll_mean3        0.52
avg_lmp_roll_std3         0.52
avg_lmp_roll_mean24       0.00
avg_lmp_roll_std24        0.00
demand_mwh_roll_mean3     0.61
demand_mwh_roll_mean24    0.06
dtype: float64
Naive baseline MAE:  7.60
Model MAE:           9.34
Model RMSE:          10.44
Improvement over baseline: -22.9%

=== Stress Score Forecast ===
Usable rows (target present): 7
Feature completeness:
hour_of_day               1.00
day_of_week               1.00
is_weekend                1.00
forecast_error_mwh        

In [3]:
import sqlite3
conn = sqlite3.connect("data/grid_stress.db")
print(conn.execute("SELECT COUNT(*) FROM fact_grid_hourly").fetchone())
conn.close()

(95,)


In [4]:
%%writefile ingest/backfill.py
from datetime import datetime, timedelta, timezone

from ingest.eia_fetch import fetch_eia_caiso
from ingest.gridstatus_fetch import fetch_caiso_lmp
from ingest.save_raw import save_raw
from ingest.load_to_db import load_eia_csv, load_gridstatus_csv


def backfill(days: int = 30):
    """
    Pulls `days` worth of historical EIA + GridStatus data in daily
    chunks and loads it into the database. Lets us build a real
    training set immediately instead of waiting for the hourly
    scheduler to accumulate data in real time.
    """
    end = datetime.now(timezone.utc)

    for i in range(days):
        day_end = end - timedelta(days=i)
        day_start = day_end - timedelta(days=1)

        eia_start = day_start.strftime("%Y-%m-%dT%H")
        eia_end = day_end.strftime("%Y-%m-%dT%H")
        gs_start = day_start.strftime("%Y-%m-%d")
        gs_end = day_end.strftime("%Y-%m-%d")

        print(f"\n--- Backfilling day {i+1}/{days}: {gs_start} to {gs_end} ---")

        try:
            df_eia = fetch_eia_caiso(start=eia_start, end=eia_end)
            path = save_raw(df_eia, source="eia")
            load_eia_csv(path)
        except Exception as e:
            print(f"EIA backfill failed for this day: {e}")

        try:
            df_gs = fetch_caiso_lmp(start=gs_start, end=gs_end)
            path = save_raw(df_gs, source="gridstatus")
            load_gridstatus_csv(path)
        except Exception as e:
            print(f"GridStatus backfill failed for this day: {e}")

    print("\nBackfill complete.")


if __name__ == "__main__":
    backfill(days=30)

Writing ingest/backfill.py


In [5]:
import sqlite3
conn = sqlite3.connect("data/grid_stress.db")
print(conn.execute("SELECT COUNT(*) FROM fact_grid_hourly").fetchone())
conn.close()

(95,)


In [6]:
import sqlite3
conn = sqlite3.connect("data/grid_stress.db")
print("raw_eia:", conn.execute("SELECT COUNT(*) FROM raw_eia").fetchone())
print("raw_gridstatus:", conn.execute("SELECT COUNT(*) FROM raw_gridstatus").fetchone())
conn.close()

raw_eia: (320,)
raw_gridstatus: (3456,)


In [7]:
%%writefile ingest/backfill.py
from datetime import datetime, timedelta, timezone

from ingest.eia_fetch import fetch_eia_caiso
from ingest.gridstatus_fetch import fetch_caiso_lmp
from ingest.save_raw import save_raw
from ingest.load_to_db import load_eia_csv, load_gridstatus_csv
from ingest.transform import load_raw_from_db, build_fact_table, load_fact_table


def backfill(days: int = 30):
    end = datetime.now(timezone.utc)

    for i in range(days):
        day_end = end - timedelta(days=i)
        day_start = day_end - timedelta(days=1)

        eia_start = day_start.strftime("%Y-%m-%dT%H")
        eia_end = day_end.strftime("%Y-%m-%dT%H")
        gs_start = day_start.strftime("%Y-%m-%d")
        gs_end = day_end.strftime("%Y-%m-%d")

        print(f"\n--- Backfilling day {i+1}/{days}: {gs_start} to {gs_end} ---")

        try:
            df_eia = fetch_eia_caiso(start=eia_start, end=eia_end)
            path = save_raw(df_eia, source="eia")
            load_eia_csv(path)
        except Exception as e:
            print(f"EIA backfill failed for this day: {e}")

        try:
            df_gs = fetch_caiso_lmp(start=gs_start, end=gs_end)
            path = save_raw(df_gs, source="gridstatus")
            load_gridstatus_csv(path)
        except Exception as e:
            print(f"GridStatus backfill failed for this day: {e}")

    # NEW: rebuild the fact table from ALL raw data now sitting in the DB
    print("\n--- Rebuilding fact_grid_hourly from all raw data ---")
    df_eia_all, df_gs_all = load_raw_from_db()
    fact = build_fact_table(df_eia_all, df_gs_all)
    inserted = load_fact_table(fact)
    print(f"Backfill complete. {inserted} new fact rows inserted.")


if __name__ == "__main__":
    backfill(days=30)

Overwriting ingest/backfill.py


In [8]:
import importlib
import ingest.backfill
importlib.reload(ingest.backfill)
from ingest.backfill import backfill

backfill(days=30)


--- Backfilling day 1/30: 2026-08-24 to 2026-08-25 ---


2026-08-25 11:15:30 - INFO - Fetching Page 1...
2026-08-25 11:15:30 - INFO - GET https://api.gridstatus.io/v1/datasets/caiso_lmp_real_time_5_min/query
2026-08-25 11:15:30 - INFO - Params: {'start_time': Timestamp('2026-08-24 00:00:00'), 'end_time': Timestamp('2026-08-25 00:00:00'), 'publish_time_start': None, 'publish_time_end': None, 'limit': None, 'page': 1, 'page_size': None, 'resample_frequency': None, 'resample_by': None, 'resample_function': None, 'publish_time': None, 'timezone': None, 'cursor': '', 'filter_column': 'location', 'filter_value': 'TH_NP15_GEN-APND,TH_SP15_GEN-APND,TH_ZP26_GEN-APND', 'filter_operator': 'in', 'return_format': 'json', 'json_schema': 'array-of-arrays'}


Saved 53 rows to data/raw\eia_20260825T054530Z.csv
raw_eia: inserted 34 new rows (19 duplicates skipped)


2026-08-25 11:15:32 - INFO - Done in 2.36 seconds. 
2026-08-25 11:15:32 - INFO - Total number of rows: 864


Saved 864 rows to data/raw\gridstatus_20260825T054532Z.csv
raw_gridstatus: inserted 864 new rows (0 duplicates skipped)

--- Backfilling day 2/30: 2026-08-23 to 2026-08-24 ---


...

Saved 861 rows to data/raw\gridstatus_20260825T085128Z.csv
raw_gridstatus: inserted 861 new rows (0 duplicates skipped)

--- Backfilling day 30/30: 2026-07-26 to 2026-07-27 ---


2026-08-25 14:21:32 - INFO - Fetching Page 1...
2026-08-25 14:21:32 - INFO - GET https://api.gridstatus.io/v1/datasets/caiso_lmp_real_time_5_min/query
2026-08-25 14:21:32 - INFO - Params: {'start_time': Timestamp('2026-07-26 00:00:00'), 'end_time': Timestamp('2026-07-27 00:00:00'), 'publish_time_start': None, 'publish_time_end': None, 'limit': None, 'page': 1, 'page_size': None, 'resample_frequency': None, 'resample_by': None, 'resample_function': None, 'publish_time': None, 'timezone': None, 'cursor': '', 'filter_column': 'location', 'filter_value': 'TH_NP15_GEN-APND,TH_SP15_GEN-APND,TH_ZP26_GEN-APND', 'filter_operator': 'in', 'return_format': 'json', 'json_schema': 'array-of-arrays'}


Saved 100 rows to data/raw\eia_20260825T085132Z.csv
raw_eia: inserted 96 new rows (4 duplicates skipped)


2026-08-25 14:21:34 - INFO - Done in 2.05 seconds. 
2026-08-25 14:21:34 - INFO - Total number of rows: 864


Saved 864 rows to data/raw\gridstatus_20260825T085134Z.csv
raw_gridstatus: inserted 864 new rows (0 duplicates skipped)

--- Rebuilding fact_grid_hourly from all raw data ---
fact_grid_hourly: inserted 626 new rows (95 duplicates skipped)
Backfill complete. 626 new fact rows inserted.


In [9]:
import sqlite3
conn = sqlite3.connect("data/grid_stress.db")
print(conn.execute("SELECT COUNT(*) FROM fact_grid_hourly").fetchone())
conn.close()

(721,)


In [10]:
%%writefile ingest/features.py
import pandas as pd
import numpy as np


def load_fact_table():
    import sqlite3
    conn = sqlite3.connect("data/grid_stress.db")
    df = pd.read_sql("SELECT * FROM fact_grid_hourly ORDER BY hour_utc", conn)
    conn.close()
    df["hour_utc"] = pd.to_datetime(df["hour_utc"])
    return df


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values("hour_utc").reset_index(drop=True)

    df["hour_of_day"] = df["hour_utc"].dt.hour
    df["day_of_week"] = df["hour_utc"].dt.dayofweek
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

    df["forecast_error_mwh"] = df["demand_mwh"] - df["day_ahead_forecast_mwh"]
    df["interchange_ratio"] = df["total_interchange_mwh"] / df["demand_mwh"]

    for lag in [1, 3, 6, 24]:
        df[f"avg_lmp_lag{lag}"] = df["avg_lmp"].shift(lag)
        df[f"demand_mwh_lag{lag}"] = df["demand_mwh"].shift(lag)

    for window in [3, 24]:
        df[f"avg_lmp_roll_mean{window}"] = df["avg_lmp"].shift(1).rolling(window, min_periods=max(2, window//4)).mean()
        df[f"avg_lmp_roll_std{window}"] = df["avg_lmp"].shift(1).rolling(window, min_periods=max(2, window//4)).std()
        df[f"demand_mwh_roll_mean{window}"] = df["demand_mwh"].shift(1).rolling(window, min_periods=max(2, window//4)).mean()

    df["target_price_next_hour"] = df["avg_lmp"].shift(-1)

    # --- Stress score: back to full 24hr window now that we have real history ---
    roll_mean_24 = df["avg_lmp"].shift(1).rolling(24, min_periods=6).mean()
    price_stress = (df["avg_lmp"] - roll_mean_24) / roll_mean_24.abs().replace(0, np.nan)

    demand_stress = (df["demand_mwh"] - df["net_generation_mwh"]) / df["demand_mwh"]

    congestion_roll = df["avg_congestion"].abs().rolling(24, min_periods=6).mean()
    volatility_stress = df["avg_congestion"].abs() / congestion_roll.replace(0, np.nan)

    df["stress_score"] = (
        price_stress.rank(pct=True) * 100 * 0.4
        + demand_stress.rank(pct=True) * 100 * 0.35
        + volatility_stress.rank(pct=True) * 100 * 0.25
    )

    df["target_stress_next_hour"] = df["stress_score"].shift(-1)

    return df


if __name__ == "__main__":
    df = load_fact_table()
    print("Raw fact table shape:", df.shape)
    featured = engineer_features(df)
    print("Featured shape:", featured.shape)
    print(featured.tail(10))

Overwriting ingest/features.py


In [11]:
import importlib
import ingest.features
importlib.reload(ingest.features)
import ingest.train_model
importlib.reload(ingest.train_model)
from ingest.train_model import train_and_evaluate
import joblib

price_model, price_metrics = train_and_evaluate("target_price_next_hour", "Price Forecast")
if price_model:
    joblib.dump(price_model, "data/price_model.pkl")

stress_model, stress_metrics = train_and_evaluate("target_stress_next_hour", "Stress Score Forecast")
if stress_model:
    joblib.dump(stress_model, "data/stress_model.pkl")


=== Price Forecast ===
Usable rows (target present): 651
Feature completeness:
hour_of_day               1.00
day_of_week               1.00
is_weekend                1.00
forecast_error_mwh        0.99
interchange_ratio         0.95
avg_lmp_lag1              0.99
avg_lmp_lag3              0.97
avg_lmp_lag6              0.95
avg_lmp_lag24             0.95
demand_mwh_lag1           0.99
demand_mwh_lag3           0.99
demand_mwh_lag6           0.99
demand_mwh_lag24          0.96
avg_lmp_roll_mean3        0.98
avg_lmp_roll_std3         0.98
avg_lmp_roll_mean24       0.99
avg_lmp_roll_std24        0.99
demand_mwh_roll_mean3     1.00
demand_mwh_roll_mean24    0.99
dtype: float64
Naive baseline MAE:  12.21
Model MAE:           7.99
Model RMSE:          23.59
Improvement over baseline: 34.6%

=== Stress Score Forecast ===
Usable rows (target present): 613
Feature completeness:
hour_of_day               1.00
day_of_week               1.00
is_weekend                1.00
forecast_error_mwh     

In [12]:
%%writefile app.py
import streamlit as st
import pandas as pd
import sqlite3
import joblib
import plotly.graph_objects as go
from datetime import datetime

from ingest.features import engineer_features

st.set_page_config(page_title="Grid Stress Monitor", layout="wide")

# ---------- Data loading ----------
@st.cache_data(ttl=300)  # refresh every 5 minutes
def load_data():
    conn = sqlite3.connect("data/grid_stress.db")
    df = pd.read_sql("SELECT * FROM fact_grid_hourly ORDER BY hour_utc", conn)
    conn.close()
    df["hour_utc"] = pd.to_datetime(df["hour_utc"])
    return df

@st.cache_resource
def load_models():
    price_model = joblib.load("data/price_model.pkl")
    stress_model = joblib.load("data/stress_model.pkl")
    return price_model, stress_model

FEATURE_COLS = [
    "hour_of_day", "day_of_week", "is_weekend",
    "forecast_error_mwh", "interchange_ratio",
    "avg_lmp_lag1", "avg_lmp_lag3", "avg_lmp_lag6", "avg_lmp_lag24",
    "demand_mwh_lag1", "demand_mwh_lag3", "demand_mwh_lag6", "demand_mwh_lag24",
    "avg_lmp_roll_mean3", "avg_lmp_roll_std3",
    "avg_lmp_roll_mean24", "avg_lmp_roll_std24",
    "demand_mwh_roll_mean3", "demand_mwh_roll_mean24",
]

df = load_data()
featured = engineer_features(df)
price_model, stress_model = load_models()

# ---------- Header ----------
st.title("⚡ CAISO Grid Stress & Price Intelligence")
st.caption(f"Live pipeline: EIA demand data + GridStatus.io real-time pricing | Last data point: {df['hour_utc'].max()}")

# ---------- Latest prediction ----------
latest = featured.iloc[[-1]]
latest_features = latest[FEATURE_COLS]

col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric("Latest Demand (MWh)", f"{latest['demand_mwh'].values[0]:,.0f}")

with col2:
    latest_price = latest["avg_lmp"].values[0]
    st.metric("Latest Avg Price ($/MWh)", f"${latest_price:,.2f}" if pd.notna(latest_price) else "Pending")

try:
    predicted_price = price_model.predict(latest_features)[0]
    with col3:
        st.metric("Predicted Next-Hour Price", f"${predicted_price:,.2f}")
except Exception:
    with col3:
        st.metric("Predicted Next-Hour Price", "N/A")

try:
    predicted_stress = stress_model.predict(latest_features)[0]
    with col4:
        risk_label = "🔴 High" if predicted_stress > 70 else "🟡 Moderate" if predicted_stress > 40 else "🟢 Low"
        st.metric("Predicted Stress Risk", risk_label, f"{predicted_stress:.0f}/100")
except Exception:
    with col4:
        st.metric("Predicted Stress Risk", "N/A")

st.divider()

# ---------- Charts ----------
recent = featured.tail(168)  # last 7 days

tab1, tab2, tab3 = st.tabs(["Price & Demand", "Grid Stress Score", "Model Info"])

with tab1:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=recent["hour_utc"], y=recent["demand_mwh"], name="Demand (MWh)", yaxis="y1"))
    fig.add_trace(go.Scatter(x=recent["hour_utc"], y=recent["avg_lmp"], name="Avg Price ($/MWh)", yaxis="y2"))
    fig.update_layout(
        yaxis=dict(title="Demand (MWh)"),
        yaxis2=dict(title="Price ($/MWh)", overlaying="y", side="right"),
        legend=dict(orientation="h", y=1.1),
        height=450,
    )
    st.plotly_chart(fig, use_container_width=True)

with tab2:
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=recent["hour_utc"], y=recent["stress_score"], name="Stress Score", fill="tozeroy"))
    fig2.add_hline(y=70, line_dash="dash", line_color="red", annotation_text="High stress threshold")
    fig2.update_layout(yaxis=dict(title="Stress Score (0-100)"), height=450)
    st.plotly_chart(fig2, use_container_width=True)

with tab3:
    st.subheader("Model Performance (held-out test set)")
    st.write("**Price Forecast Model**: 34.6% MAE improvement over naive baseline (651 training hours)")
    st.write("**Stress Score Model**: 65.7% MAE improvement over naive baseline (613 training hours)")
    st.caption("Both models are HistGradientBoostingRegressor, trained on lag/rolling/calendar features from EIA + GridStatus.io data.")

    importances = pd.Series(price_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
    st.subheader("Top Price Model Features")
    st.bar_chart(importances.head(8))

Writing app.py
